[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leehart3000/zarrextract/blob/main/notebooks/ome_zarr_idr0051.ipynb)

# Light-sheet microscopy (OME-Zarr) with zarrextract

This notebook reads small pieces of an OME-Zarr 0.5 image from the Image Data Resource (IDR): the intensity at one pixel over all 79 time points, and one 2D slice of the half-resolution image. The store is Zarr v3 and is read over HTTPS. Nothing is downloaded except the chunks that overlap each selection.

The image is a light-sheet movie of the neuromesodermal progenitor zone in the tailbud of a zebrafish embryo, with a nuclear marker. It has the dimensions t, c, z, y and x.

- Licence: CC BY 4.0. Copyright: Steventon et al.
- Study: IDR idr0051, "In Toto Cell Tracking using Lightsheet Microscopy of the Neuromesodermal Progenitor Zone of the Zebrafish Tailbud from the 21st Somite Stage". Data DOI: https://doi.org/10.17867/10000117
- Publication: Attardi A, Fulton T, Florescu M, Shah G, Muresan L, Lenz MO, Lancaster C, Huisken J, van Oudenaarden A, Steventon B (2018), Development, https://doi.org/10.1242/dev.166728
- Also cite the Image Data Resource: Williams E. et al. (2017), Nature Methods 14, 775-781, https://doi.org/10.1038/nmeth.4326

## Install

Reading a store over HTTPS needs `fsspec` and `aiohttp`. Colab already has both, so installing `zarrextract` is enough.

In [ ]:
%pip install -q zarrextract

In [ ]:
import time
from datetime import UTC, datetime

import matplotlib.pyplot as plt
import numpy as np
import zarr

import zarrextract as zx

STORE = (
    "https://livingobjects.ebi.ac.uk/idr/zarr/v0.5/idr0051/"
    "180712_H2B_22ss_Courtney1_20180712-163837_p00_c00_preview.zarr"
)
FULL = "0/0"  # full resolution
HALF = "0/1"  # half resolution in y and x

accessed = datetime.now(UTC).strftime("%Y-%m-%d")
print(
    "Image Data Resource idr0051 (https://doi.org/10.17867/10000117), "
    "CC BY 4.0, Steventon et al., "
    f"accessed on {accessed}"
)

## Look at the store

`describe()` lists the arrays without reading any data. This server cannot list folders and the store has no consolidated metadata, so `describe()` cannot find the arrays and raises `StoreOpenError`. The message says what to do.

In [ ]:
try:
    zx.describe(STORE)
except zx.StoreOpenError as error:
    print("StoreOpenError:", error)

`extract()` does not need a listing. It only needs the path of the array, and OME-Zarr stores the resolution levels as arrays `0/0` (full resolution), `0/1` (half resolution) and so on. We can open one with `zarr` to see its shape, its chunks and its dimension names.

In [ ]:
group = zarr.open_group(STORE, mode="r")
for path in (FULL, HALF):
    array = group[path]
    print(path, array.metadata.dimension_names, array.shape, array.dtype)

## What the index positions mean

This store has no coordinate arrays for t, c, z, y and x, so zarrextract uses the index positions: `y=150` means the 151st row. The OME metadata, in the attributes of the group `0`, says what the positions stand for. It lists the axes and, for each resolution level, the scale of each axis.

In [ ]:
multiscale = group["0"].attrs["ome"]["multiscales"][0]
names = [axis["name"] for axis in multiscale["axes"]]

print("axes:")
for axis in multiscale["axes"]:
    unit = axis.get("unit", "not given")
    print(f"  {axis['name']}: type {axis['type']}, unit {unit}")

scales = {}
print("scale of each axis, per resolution level:")
for dataset in multiscale["datasets"]:
    (transform,) = dataset["coordinateTransformations"]
    scales[dataset["path"]] = dict(zip(names, transform["scale"], strict=True))
    print(f"  0/{dataset['path']}:", scales[dataset["path"]])

The scale of the full-resolution level is 1 on every axis and the axes have no unit. This image has no physical sizes in its metadata, so the positions are pixels and time points, and we cannot give them in micrometres or minutes. The half-resolution level has a scale of 2 in y and x: one of its pixels covers 2 full-resolution pixels. We use the scale to put the slice below in full-resolution pixels.

## A time series at one pixel

We pick the full-resolution pixel `y=150`, `x=100` in the middle plane `z=100` (of 201) and the only channel, `c=0`. It has signal: its intensity stays well above the background (a median near 160 in this plane) at every one of the 79 time points. A `(start, stop)` pair picks every value from start to stop, inclusive, so `t=(0, 78)` is the whole movie.

In [ ]:
start = time.perf_counter()
series = zx.extract(STORE, FULL, t=(0, 78), c=0, z=100, y=150, x=100)
elapsed = time.perf_counter() - start

print(f"{series.num_rows} rows in {elapsed:.1f} s")
print(series.slice(0, 3).to_pylist())

In [ ]:
frames = series["t"].to_numpy()
intensity = series[FULL].to_numpy()

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(frames, intensity, color="#2a6fb0", linewidth=2)
ax.set_xlabel("Time point")
ax.set_ylabel("Intensity (counts)")
ax.set_title("One pixel (z=100, y=150, x=100) over 79 time points")
ax.grid(alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## One 2D slice

A slice at `t=0`, `c=0` and the middle plane `z=100`, from the half-resolution level `0/1`. Leaving out `y` and `x` keeps them whole. The table has one row per pixel in C order (`x` changes fastest), so a reshape turns it back into an image.

In [ ]:
start = time.perf_counter()
plane = zx.extract(STORE, HALF, t=0, c=0, z=100)
elapsed = time.perf_counter() - start
print(f"{plane.num_rows} rows in {elapsed:.1f} s")

The axes show full-resolution pixels: the scale of the level multiplies the index positions. The cross marks the pixel of the time series.

In [ ]:
n_y = np.unique(plane["y"].to_numpy()).size
n_x = np.unique(plane["x"].to_numpy()).size
image = plane[HALF].to_numpy().reshape(n_y, n_x)

scale = scales["1"]
extent = (0, n_x * scale["x"], n_y * scale["y"], 0)  # left, right, bottom, top

fig, ax = plt.subplots(figsize=(6, 5.5))
shown = ax.imshow(image, cmap="gray", extent=extent)
ax.plot(100 + 0.5, 150 + 0.5, marker="+", color="#e8a33d", markersize=14)
fig.colorbar(shown, ax=ax, label="Intensity (counts)")
ax.set_xlabel("x (full-resolution pixels)")
ax.set_ylabel("y (full-resolution pixels)")
ax.set_title("Time point 0, z=100, half resolution")
plt.show()

## Why this is cheap

zarrextract reads only the chunks the selection needs. That matters for light-sheet volumes, which are large: this array is split into shards of one whole volume (one time point), and each shard holds one chunk per z plane. The time series above needs one plane of each of the 79 time points. The cell below compares what that touches with the whole array.

In [ ]:
array = group[FULL]
itemsize = array.dtype.itemsize
chunk_mb = np.prod(array.chunks) * itemsize / 1e6
whole_mb = np.prod(array.shape) * itemsize / 1e6

print("chunks:", array.chunks, "shards:", array.shards)
print(f"time series: 79 chunks of {chunk_mb:.2f} MB = {79 * chunk_mb:.0f} MB")
print(f"whole array: {whole_mb:,.0f} MB")